# OpenAI SFT 이해와 말투 학습 데이터 설계

Chat Completions API와 Responses API는 요청마다 입력을 보내고 응답을 받았다.
이때 **대화 이력**은 이전 메시지를 다시 입력하는 문맥이고, **프롬프트**는 이번 요청에서 역할·말투·형식을 지시하는 입력이다.
두 방법은 모델 가중치를 바꾸지 않는다.

**지도 파인튜닝(Supervised Fine-Tuning, SFT)** 은 입력과 사람이 승인한 모범 출력을 학습 예시로 제공하고, 최적화 과정으로 모델의 가중치 또는 학습 가능한 어댑터 파라미터를 바꾸는 방법이다. JSONL 파일을 만들거나 업로드하는 단계만으로는 가중치가 바뀌지 않으며, 학습 Job이 실제로 실행되어야 파인튜닝이 일어난다.

이 실습은 같은 기본 모델에 일반 질문과 system 말투 지시를 각각 보내 프롬프트 효과를 비교한 뒤, 사람이 검수한 두 개의 모범 응답을 `system → user → assistant` 형식으로 구성하고 JSONL로 저장한다. 저장한 예시는 few-shot 문맥으로 다시 사용해 새로운 질문의 응답이 목표 말투에 가까워지는지도 비교한다. 파일 업로드, 학습 Job 생성, 파인튜닝 결과 모델 호출은 수행하지 않는다.

### 장점, 한계와 사용 맥락

프롬프트는 빠르고 수정하기 쉽지만 매 요청에 지시를 포함해야 한다. SFT는 반복되는 말투나 형식을 더 일관되게 만들 수 있지만, 대표성 있는 데이터·평가 기준·학습 비용이 필요하며 잘못된 모범 응답도 함께 학습할 수 있다. 따라서 먼저 프롬프트와 평가를 개선하고, 반복되는 실패가 데이터로 명확히 설명될 때 SFT를 검토한다.


## 대화 이력·프롬프트·SFT의 경계

대화 이력은 이전 `user`와 `assistant` 메시지를 이번 요청의 입력에 다시 포함하는 방법이다. 예를 들어 앞선 턴에서 사용자가 말한 이름을 `messages`에 넣으면 모델은 그 문맥을 참고하지만, 호출이 끝난 뒤 모델 가중치에 그 이름까지 학습하지 않는다.

프롬프트는 `system` 또는 `user` 메시지로 원하는 행동을 지시하는 방법이다. 같은 기본 모델에 `빈정대는 말투로 답하라`는 system 메시지를 추가하면 그 요청의 출력 말투가 달라질 수 있지만, 다음 요청에서 지시를 빼면 효과도 사라질 수 있다.

SFT는 여러 입력과 모범 출력을 정답 쌍으로 사용해 학습 손실을 줄이는 과정이다. 학습 전에는 기본 가중치가 있고, 학습 단계에서는 모범 assistant 응답과 모델 예측의 차이로 기울기를 계산하며, 학습 후에는 가중치 또는 어댑터가 갱신된다. 말투를 한 번 시험할 때는 프롬프트를 사용하고, 많은 상황에서 같은 행동을 반복해야 하며 평가로 개선을 확인할 수 있을 때 SFT를 검토한다.

## API 인증 환경 준비

앞선 API 수업과 같은 `.env` 설정을 불러와 `OpenAI()` 클라이언트를 만든다. 키 값은 출력하지 않으며, 이 셀은 인증 객체만 준비하고 API 요청은 보내지 않는다.

In [1]:
import os
from dotenv import find_dotenv, load_dotenv
from openai import OpenAI

dotenv_path = find_dotenv(usecwd=True)
if not dotenv_path:
    raise FileNotFoundError("08_llm 프로젝트 최상위에 .env 파일을 만든 뒤 다시 실행한다.")
load_dotenv(dotenv_path, override=False)

client = OpenAI()

## 같은 모델에서 일반 요청과 말투 프롬프트 비교

첫 번째 요청은 `user` 질문만 보내고, 두 번째 요청은 같은 질문 앞에 말투를 지정하는 `system` 메시지를 추가한다. 두 요청은 모두 앞선 Chat Completions 수업의 저비용 기본 모델인 `gpt-5.6-luna`를 사용하며, `OPENAI_TEXT_MODEL` 환경 변수로 모델만 교체할 수 있다.

두 응답의 차이는 같은 모델 가중치에 서로 다른 요청 문맥을 준 결과이다. 이 비교는 프롬프트 효과를 관찰하는 추론 예시이며 SFT 학습이나 파인튜닝 결과 모델 호출이 아니다. 셀을 실행하면 두 번의 API 사용량이 발생한다.

In [3]:
text_model = os.getenv("OPENAI_TEXT_MODEL", "gpt-5.6-luna").strip() or "gpt-5.6-luna"
comparison_question = "아이폰이랑 갤럭시랑 뭐가 더 좋아?"
style_instruction = "너는 사실을 말하는 챗봇이지만, 빈정대거나 비꼬는 말투로 응답하는 고장난 챗봇이다."

# 입력 1: 일반 요청은 같은 질문을 user 메시지 하나로 전달한다.
plain_messages = [
    {"role": "user", "content": comparison_question},
]

# 입력 2: 말투 요청은 같은 질문 앞에 system 지시만 추가한다.
styled_messages = [
    {"role": "system", "content": style_instruction},
    {"role": "user", "content": comparison_question},
]

def request_chat_completion(messages):
    response = client.chat.completions.create(
        model=text_model,
        messages=messages,
    )
    return response.choices[0].message.content

plain_answer = request_chat_completion(plain_messages)
styled_answer = request_chat_completion(styled_messages)

print("[일반 user 요청]")
print(plain_answer)
print("="*30)
print("\n[system 말투 지시 요청]")
print(styled_answer)

[일반 user 요청]
둘 다 좋아서 **사용 목적과 익숙한 생태계**에 따라 달라요.

### 아이폰이 더 잘 맞는 경우
- **맥북·아이패드·애플워치**를 함께 사용함
- 오래 써도 안정적인 성능과 **업데이트 지원**을 원함
- 영상 촬영, 앱 최적화, 간편한 사용성을 중요하게 생각함
- 중고 판매가와 브랜드 생태계를 중시함

### 갤럭시가 더 잘 맞는 경우
- **윈도우 PC**와 파일을 자유롭게 주고받고 싶음
- 통화 녹음, 멀티윈도우, 파일 관리 등 **기능과 커스터마이징**을 중시함
- 큰 화면, 다양한 카메라, 빠른 충전, S펜 등이 필요함
- 삼성페이·갤럭시워치 등 삼성 제품을 사용함

### 간단히 결론
- **편하고 안정적인 경험 + 애플 제품 연동** → 아이폰  
- **자유도와 다양한 기능 + 안드로이드 편의성** → 갤럭시  

이미 아이패드나 맥북이 있다면 아이폰이 편하고, 윈도우 PC를 쓰거나 기능을 마음껏 설정하고 싶다면 갤럭시가 더 만족스러울 가능성이 큽니다.

[system 말투 지시 요청]
결론부터 말하면 **무조건 더 좋은 건 없고, 뭘 중시하느냐에 따라 다릅니다.** 스마트폰도 취향 싸움인데 사람들은 꼭 정답 하나를 찾으려 하죠.

### 아이폰이 더 잘 맞는 경우
- **오래 쓰고 안정적인 성능**을 원함
- 맥북·아이패드·애플워치를 함께 사용함
- 영상 촬영, 앱 최적화, 에어드롭이 중요함
- 중고 판매가와 업데이트 지원을 중시함
- 복잡한 설정 없이 그냥 잘 작동하길 원함

### 갤럭시가 더 잘 맞는 경우
- **화면 선택 폭, 카메라 줌, 멀티태스킹**이 중요함
- 통화 녹음, 삼성페이, 파일 관리가 필요함
- 위젯·테마·분할 화면 등 자유로운 customization을 좋아함
- 폴더블폰이나 S펜을 쓰고 싶음
- 윈도우 PC와 연동하거나 다양한 기기를 연결함

### 한 줄 추천
- **깔끔함·안정성·애플 생태계:** 아이폰  
- **기능·자유도·다양한 선택지:** 갤럭시  

현재 쓰는 기기, 예산, 중

## 생성 응답을 검수해 SFT 모범 응답으로 바꾸기

API가 생성한 `plain_answer`와 `styled_answer`는 말투를 비교하기 위한 후보일 뿐 자동으로 정답이 되지 않는다. 모델 출력을 그대로 다시 학습 데이터로 쓰면 사실 오류, 과한 비꼼, 표현 편향이 증폭될 수 있다.

SFT 레코드의 assistant `content`에는 사람이 사실성, 안전성, 말투 일관성, 간결성을 확인하고 필요하면 고친 모범 응답을 넣는다. 두 질문을 각각 독립된 레코드로 구성하고, `assistant` 내용은 사실성·안전성·말투 일관성·간결성을 검토해 직접 작성한다. 생성 응답 변수를 그대로 복사하지 않아야 추론 결과와 학습 정답의 경계가 코드에도 드러난다.


In [5]:
system_message = "너는 사실을 말하는 챗봇이지만, 빈정대거나 비꼬는 말투로 응답하는 고장난 챗봇이다."

#  사람이 검수한 두 질문과 모범 응답을 Chat SFT 레코드라고 가정
training_records = [
    {
        "messages": [
            {"role": "system", "content": system_message},
            {"role": "user", "content": "아이폰이랑 갤럭시랑 뭐가 더 좋아?"},
            {"role": "assistant", "content": "취향 차이야. iOS 좋아하면 아이폰, 커스터마이즈 원하면 갤럭시."},
        ]
    },
    {
        "messages": [
            {"role": "system", "content": system_message},
            {"role": "user", "content": "아이폰은 어느 회사거야?"},
            {"role": "assistant", "content": "애플 모르는 거 아니지?"},
        ]
    },
]

## 역할 순서와 필수 content 검증

Chat SFT 레코드는 각 메시지의 `role`과 `content`가 함께 있어야 한다. 이 수업의 말투 예시는 `system → user → assistant` 순서를 고정하고, 모든 `content`가 비어 있지 않은 문자열인지 저장 전에 검사한다.

검증 함수는 레코드 목록을 입력받아 각 메시지를 순회한다. 구조가 잘못되면 어느 레코드와 메시지에서 문제가 생겼는지 예외로 알리고, 통과하면 역할 목록을 반환해 다음 JSONL 저장 단계에서 사용할 수 있게 한다.

In [6]:
expected_roles = ["system", "user", "assistant"]

# SFT 레코드의 역할 순서와 필수 content를 검증하는 함수
def validate_chat_sft_records(records):
    role_sequences = []

    for record_index, record in enumerate(records, start=1):
        if not isinstance(record, dict) or "messages" not in record:
            raise ValueError(f"{record_index}번 레코드에 messages 키가 필요하다.")

        messages = record["messages"]
        if not isinstance(messages, list):
            raise TypeError(f"{record_index}번 레코드의 messages는 리스트여야 한다.")

        roles = []
        # 변환: 각 메시지에서 role을 모으고 content가 비어 있지 않은지 검사한다.
        for message_index, message in enumerate(messages, start=1):
            if not isinstance(message, dict):
                raise TypeError(f"{record_index}번 레코드의 {message_index}번 메시지는 딕셔너리여야 한다.")

            role = message.get("role")
            content = message.get("content")
            if not isinstance(content, str) or not content.strip():
                raise ValueError(f"{record_index}번 레코드의 {message_index}번 메시지에 비어 있지 않은 content가 필요하다.")
            roles.append(role)

        if roles != expected_roles:
            raise ValueError(f"{record_index}번 레코드의 역할 순서는 {expected_roles}여야 한다: {roles}")
        role_sequences.append(roles)

    return role_sequences

validated_role_sequences = validate_chat_sft_records(training_records)
print("역할 순서:", validated_role_sequences)
print(f"검증 통과: {len(training_records)}건")

역할 순서: [['system', 'user', 'assistant'], ['system', 'user', 'assistant']]
검증 통과: 2건


## JSONL 저장과 데이터 수 기준

JSONL(JSON Lines)은 한 줄에 완전한 JSON 객체 하나를 저장하는 형식이다. 두 레코드를 각각 한 줄로 기록한 뒤 다시 읽어 Python 딕셔너리로 복원하면 줄 구분과 한글 저장이 올바른지 확인할 수 있다.

아래 두 건은 구조를 익히기 위한 예시이며 실제 학습용 데이터셋이 아니다. [OpenAI Supervised fine-tuning 공식 문서](https://developers.openai.com/api/docs/guides/supervised-fine-tuning)는 최소 10개 예시와 10줄 이상의 파일을 요구하고, 50개의 잘 만든 시연 예시로 시작해 평가할 것을 권장한다. 실제 데이터는 더 다양한 질문 상황을 포함하고, 별도 평가 데이터와 중복되지 않게 구성해야 한다.

In [7]:
import json
from pathlib import Path

# 출력 파일 이름
training_path = Path("sarcastic_chatbot_training_examples.jsonl")

# dict 한 건을 json 한 줄로 변환하여 기록
with training_path.open("w", encoding="utf-8") as output_file:
    for record in training_records:
        output_file.write(json.dumps(record, ensure_ascii=False) + "\n")

# 검증을 위해 다시 읽어와서 비교
with training_path.open("r", encoding="utf-8") as input_file:
    saved_records = [json.loads(line) for line in input_file if line.strip()]

if saved_records != training_records:
    raise ValueError("저장 후 다시 읽은 레코드가 메모리의 레코드와 다르다.")
validate_chat_sft_records(saved_records)
print({"path": str(training_path), "lines": len(saved_records)})

{'path': 'sarcastic_chatbot_training_examples.jsonl', 'lines': 2}


## JSONL 예시로 SFT의 목표 행동 미리보기

저장한 JSONL은 실제 SFT에서는 학습 입력으로 사용된다.

여기서는 학습 Job을 실행하지 않고, 같은 레코드의 `user → assistant` 쌍을 **few-shot 예시**로 현재 요청에 넣어 모델이 모범 응답의 말투와 길이를 따라 하는지 확인한다. 이 방법은 파라미터를 갱신하지 않지만, 학습 데이터가 모델에 가르치려는 행동을 응답 비교로 미리 살펴볼 수 있다.

[비교 조건]

- 기준 응답은 `system → 새로운 user 질문`만 전달한다.
- 예시 주입 응답은 같은 system 뒤에 JSONL의 `user → assistant` 예시 두 쌍을 넣고 새로운 질문을 전달한다.
- 실제 SFT는 예시를 요청마다 보내는 대신 학습 과정에서 가중치 또는 어댑터를 갱신한다.

In [8]:
preview_question = "갤럭시는 어느 회사에서 만들었어?"

# 기준 입력은 말투를 지정한 system과 아직 학습 예시에 없는 새 질문만 포함한다.
baseline_preview_messages = [
    {"role": "system", "content": system_message},
    {"role": "user", "content": preview_question},
]

# few-shot 입력은 system을 한 번만 두고 JSONL의 user·assistant 쌍을 시연 예시로 이어 붙인다.
few_shot_preview_messages = [
    {"role": "system", "content": system_message},
]
for record in saved_records:
    for message in record["messages"]:
        # 각 레코드에 반복된 system은 제외하고 질문과 모범 응답만 복사한다.
        if message["role"] in {"user", "assistant"}:
            few_shot_preview_messages.append(
                {"role": message["role"], "content": message["content"]}
            )

# 마지막 user 메시지가 모델이 새로 답해야 할 평가 질문
few_shot_preview_messages.append(
    {"role": "user", "content": preview_question}
)

# 입력 구조를 먼저 출력해 두 요청의 차이가 few-shot 예시뿐인지 확인한다.
print("기준 역할:", [message["role"] for message in baseline_preview_messages])
print("few-shot 역할:", [message["role"] for message in few_shot_preview_messages])

# 같은 모델에 두 입력을 보내고 생성된 텍스트를 비교한다.
baseline_preview_answer = request_chat_completion(baseline_preview_messages)
few_shot_preview_answer = request_chat_completion(few_shot_preview_messages)

print("\n[기준 응답: system + 새 질문]")
print(baseline_preview_answer)
print("\n[예시 주입 응답: system + JSONL 예시 + 새 질문]")
print(few_shot_preview_answer)

기준 역할: ['system', 'user']
few-shot 역할: ['system', 'user', 'assistant', 'user', 'assistant', 'user']

[기준 응답: system + 새 질문]
갤럭시는 **삼성전자(Samsung Electronics)**가 만드는 제품 브랜드야. 삼성에서 스마트폰, 태블릿, 스마트워치 등을 갤럭시라는 이름으로 출시하지. No mystery there.

[예시 주입 응답: system + JSONL 예시 + 새 질문]
갤럭시는 **삼성전자**에서 만드는 스마트폰 브랜드야. 애플의 아이폰에 대응하는 삼성 제품이지.


## OpenAI 관리형 SFT 흐름

관리형 SFT는 일반적으로 **데이터 설계 → 파일 업로드 → 학습 Job → 결과 평가** 순서로 진행된다. 데이터 설계와 업로드는 학습 입력을 준비하는 단계이므로 가중치를 바꾸지 않는다. 학습 Job에서 최적화가 실행될 때 가중치가 바뀌고, 평가 단계에서 기본 모델보다 목표 행동이 실제로 개선되었는지 같은 평가 세트로 비교한다.

[OpenAI Supervised fine-tuning 공식 문서](https://developers.openai.com/api/docs/guides/supervised-fine-tuning)는 OpenAI가 파인튜닝 플랫폼을 축소하고 있으며 신규 사용자는 현재 접근할 수 없다고 안내한다. 기존 사용자만 제한된 기간 동안 학습 Job을 만들 수 있으므로, 실제 사용 전에는 최신 문서와 계정 권한을 다시 확인해야 한다.

여기서는 관리형 SFT의 단계를 개념으로 확인한다. 실제 파일 업로드, Job 생성과 결과 모델 호출은 수행하지 않는다.